In [0]:
# Databricks notebook source
# COMMAND ----------
# DBTITLE 1,Initialize Logger & Task Context
%run ./00_audit_utilsnb.ipynb

# COMMAND ----------
import traceback
from datetime import datetime
from pyspark.sql.functions import (
    col, current_timestamp, concat_ws, coalesce, lit, 
    md5, regexp_replace, to_date, trim, upper, when
)

Warning you are using the ipython `%run` line magic. To use the databricks `%run` cell magic make sure that the magic is at the very start of the cell.

In [0]:
# Starting runlog entries
get_run_id = spark.sql("SELECT COALESCE(MAX(run_id), 0) + 1 AS next_id FROM shprod.audit_runlog").collect()[0]["next_id"]
logger = PipelineLogger(spark=spark, pipeline_name="NYC_PAYROLL_BATCH_PVT_PIPELINE", run_id=get_run_id)
logger.open_runlog()

In [0]:
# Export run_id to downstream Databricks workflow tasks
dbutils.jobs.taskValues.set(key="pipeline_run_id", value=logger.run_id)

In [0]:
# COMMAND ----------
# DBTITLE 1,Ingest, Validate, and Route Records
step_start = datetime.now()
source_file_path = "/Volumes/workspace/shprod/payrolldata/data/payrolldata.csv"

In [0]:
# Checking for raw CSV batch
raw_f = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(source_file_path)
)
rows_read = raw_f.count()
print(rows_read)
#raw_f.select("Work Location Borough").show()
raw_f.printSchema()

509525
root
 |-- Fiscal Year: string (nullable = true)
 |-- Agency Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Mid Init: string (nullable = true)
 |-- Agency Start Date: string (nullable = true)
 |-- Work Location Borough: string (nullable = true)
 |-- Title Description: string (nullable = true)
 |-- Leave Status as of June 30: string (nullable = true)
 |-- Base Salary: string (nullable = true)
 |-- Pay Basis: string (nullable = true)
 |-- Regular Hours: string (nullable = true)
 |-- Regular Gross Paid: string (nullable = true)
 |-- OT Hours: string (nullable = true)
 |-- Total OT Paid: string (nullable = true)
 |-- Total Other Pay: string (nullable = true)



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, lit, concat_ws, upper, trim, to_date, 
    regexp_replace, when, current_timestamp, 
    xxhash64, abs, row_number
)

try:
    # 1. Read raw CSV batch
    raw_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(source_file_path)
    )
    rows_read = raw_df.count()

    # 2. Standardize & generate deterministic surrogate key (pid)
    standardized_df = raw_df.select(
    # Surrogate Primary Key
    abs(xxhash64(
        concat_ws(
            "||",
            coalesce(upper(trim(col("First Name"))), lit("")),
            coalesce(upper(trim(col("Last Name"))), lit("")),
            coalesce(upper(trim(col("Agency Name"))), lit("")),
            coalesce(upper(trim(col("Title Description"))), lit("")),
            coalesce(upper(trim(col("Base Salary"))), lit("")),
            coalesce(upper(trim(col("Base Salary"))), lit("")),
            coalesce(to_date(col("Agency Start Date"), "MM/dd/yyyy").cast("string"), lit(""))
        )
    )).alias("pid"),
    
    # 16 Raw Columns Mapped & Typed
    col("Fiscal Year").cast("int").alias("fiscal_year"),
    upper(trim(col("Agency Name"))).alias("agency_name"),
    upper(trim(col("First Name"))).alias("first_name"),
    upper(trim(col("Last Name"))).alias("last_name"),
    upper(trim(col("Mid Init"))).alias("mid_name"),
    to_date(col("Agency Start Date"), "MM/dd/yyyy").alias("agency_start_date"),
    upper(trim(col("Work Location Borough"))).alias("work_location_borough"),
    upper(trim(col("Title Description"))).alias("title_description"),
    upper(trim(col("Leave Status as of June 30"))).alias("leave_status_as_of_june_30"),
    regexp_replace(col("Base Salary"), r"[\$,]", "").cast("decimal(12,2)").alias("base_salary"),
    upper(trim(col("Pay Basis"))).alias("pay_basis"),
    regexp_replace(col("Regular Hours"), r"[,]", "").cast("decimal(10,2)").alias("regular_hours"),
    regexp_replace(col("Regular Gross Paid"), r"[\$,]", "").cast("decimal(12,2)").alias("regular_gross_paid"),
    regexp_replace(col("OT Hours"), r"[,]", "").cast("decimal(10,2)").alias("ot_hours"),
    regexp_replace(col("Total OT Paid"), r"[\$,]", "").cast("decimal(12,2)").alias("total_ot_paid"),
    regexp_replace(col("Total Other Pay"), r"[\$,]", "").cast("decimal(12,2)").alias("total_other_pay"),
    
    # Audit Column
    lit(None).cast("date").alias("active_dt"),
    lit(logger.run_id).alias("run_id")
)
    # 3. Validation Rules
    valid_cond = (
        col("pid").isNotNull() &
        col("fiscal_year").isNotNull() &
        (col("first_name").isNotNull() | col("last_name").isNotNull()) &
        col("title_description").isNotNull() &
        (col("base_salary") >= 0)
    )

    # 4. Filter invalid rows to quarantine
    invalid_df = (
        standardized_df
        .filter(~valid_cond)
        .withColumn(
            "rejection_reason",
            when(col("fiscal_year").isNull(), lit("NULL or Invalid Fiscal Year"))
            .when(col("base_salary") < 0, lit("Negative Base Salary"))
            .when(col("title_description").isNull(), lit("NULL Title Description"))
            .otherwise(lit("Missing Name or PID"))
        )
        .withColumn("quarantined_at", current_timestamp())
    )

    # 5. Deduplicate valid rows on PID (keep first occurrence)
    valid_df = standardized_df.filter(valid_cond)
    dedup_window = Window.partitionBy("pid").orderBy(col("agency_start_date").asc_nulls_last())

    ranked_valid_df = valid_df.withColumn("rn", row_number().over(dedup_window))

    clean_df = ranked_valid_df.filter(col("rn") == 1).drop("rn")
    
    duplicate_df = (
        ranked_valid_df
        .filter(col("rn") > 1)
        .drop("rn")
        .withColumn("rejection_reason", lit("Duplicate Identifier - Dropped"))
        .withColumn("quarantined_at", current_timestamp())
    )

    # Combine all rejections
    full_quarantine_df = invalid_df.unionByName(duplicate_df)
    rows_quarantined = full_quarantine_df.count()
    if rows_quarantined > 0:
        full_quarantine_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("shprod.payroll_quarantine")

    # 6. Save clean batch to staging
    rows_clean = clean_df.count()
    clean_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("shprod.stg_payroll_employee_nyc")
    # 6. Update Active date
    logger.update_active_dt()

    # 7. Log Task Execution
    logger.log_step(
        step_name="01_INGEST_AND_VALIDATE",
        source_table="payroll.csv",
        target_table="stg_payroll_employee_nyc",
        error_table="payroll_quarantine",
        status="SUCCESS",
        rows_read=rows_read,
        rows_inserted=rows_clean,
        rows_quarantined=rows_quarantined,
        start_time=step_start,
        end_time=datetime.now()
    )
    logger.close_runlog()

except Exception as e:
    err = str(traceback.format_exc()).replace("'", "\"")[:1000]
    logger.log_step(
        step_name="01_INGEST_AND_VALIDATE",
        source_table="payroll.csv",
        target_table="stg_payroll_employee_nyc",
        error_table="payroll_quarantine",
        status="FAILED",
        start_time=step_start,
        end_time=datetime.now(),
        error_msg=err
    )
    logger.close_runlog(status="FAILED")
    raise e

In [0]:
# 8. Update History table
hdf=spark.sql("select * from shprod.stg_payroll_employee_nyc")
#hdf.show()
hdf.write.format("delta").mode("append").saveAsTable("shprod.hist_payroll_employee_nyc")
